Exercise 6 — Workout Goal to Weekly Plan Pipeline
Build a 2-step pipeline. Step 1 extracts fitness goal details (goal type, current fitness level, days available per week, equipment access) from a free-form fitness goal description. Step 2 generates a structured weekly workout plan scaled to the extracted availability, using only the extracted details from Step 1, not the original description.

Input: fitness_goal_text, days_per_week

Output: the extracted goal details from Step 1, and the weekly workout plan from Step 2

Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (fields: goal_type, fitness_level, days_available, equipment_access)

Step 2 — Template: ChatPromptTemplate (system persona: "certified personal trainer")  |  Parser: PydanticOutputParser (field: list of day-by-day plan entries)

In [1]:

from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser


In [2]:

class FitnessDetails(BaseModel):
    goal_type: str = Field(
        description="The person's main fitness goal"
    )
    fitness_level: str = Field(
        description="Beginner, intermediate, or advanced"
    )
    days_available: int = Field(
        description="Number of workout days available per week"
    )
    equipment_access: str = Field(
        description="Equipment available to the person"
    )


ex6_fitness_parser = PydanticOutputParser(
    pydantic_object=FitnessDetails
)


In [3]:

ex6_fitness_prompt = PromptTemplate(
    template="""
You are a fitness information extraction assistant.

Extract the person's fitness details from the text.

Fitness goal description:
{fitness_goal_text}

The user explicitly has {days_per_week} workout days available
per week. Treat this number as authoritative.

Extract:
- goal_type
- fitness_level
- days_available
- equipment_access

If the fitness level or equipment is not specified, say
"Not specified" rather than inventing details.

{format_instructions}
""",
    input_variables=["fitness_goal_text", "days_per_week"],
    partial_variables={
        "format_instructions":
            ex6_fitness_parser.get_format_instructions()
    }
)


In [4]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [5]:

ex6_step1_chain = (
    ex6_fitness_prompt
    | llm
    | ex6_fitness_parser
)


In [6]:

fitness_goal_text = (
    "I am a beginner who wants to build muscle at home. "
    "I have a pair of dumbbells."
)

fitness_details = ex6_step1_chain.invoke({
    "fitness_goal_text": fitness_goal_text,
    "days_per_week": 4
})

print(fitness_details)
print(fitness_details.model_dump_json(indent=2))


goal_type='Build muscle' fitness_level='Beginner' days_available=4 equipment_access='pair of dumbbells'
{
  "goal_type": "Build muscle",
  "fitness_level": "Beginner",
  "days_available": 4,
  "equipment_access": "pair of dumbbells"
}


In [7]:
print(fitness_details.goal_type)
print(fitness_details.fitness_level)
print(fitness_details.days_available)
print(fitness_details.equipment_access)

Build muscle
Beginner
4
pair of dumbbells


## Step 2 — Template: ChatPromptTemplate (system persona: "escalation-readiness reviewer")  |  Parser: PydanticOutputParser (fields: missing_fields: list, is_ready_to_escalate: bool)



In [8]:

class WorkoutDay(BaseModel):
    day: str = Field(
        description="Day label, such as Day 1 or Monday"
    )
    focus: str = Field(
        description="Main focus of the workout"
    )
    exercises: list[str] = Field(
        description="List of exercises for the day"
    )
    duration_minutes: int = Field(
        description="Approximate total workout duration in minutes"
    )
    notes: str = Field(
        description="Warm-up, rest, or relevant safety notes"
    )


class WeeklyWorkoutPlan(BaseModel):
    plan: list[WorkoutDay] = Field(
        description="The day-by-day weekly workout plan"
    )


ex6_plan_parser = PydanticOutputParser(
    pydantic_object=WeeklyWorkoutPlan
)


In [9]:

ex6_plan_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a careful, knowledgeable personal trainer.

Create a practical weekly workout plan based on the
provided structured fitness details.

Requirements:
- Match the person's fitness level, goal, available days,
  and equipment.
- Include exactly as many workout days as days_available.
- Label the sessions Day 1, Day 2, and so on.
- Keep the plan realistic and allow recovery between sessions.
- Do not prescribe equipment the person does not have.
- Include sensible warm-up and safety notes.
- Do not invent medical information about the person.
- Return the result in the required structured format.

{format_instructions}
"""
    ),
    (
        "human",
        """
Here are the extracted fitness details:

{fitness_details}
"""
    )
]).partial(
    format_instructions=ex6_plan_parser.get_format_instructions()
)


In [10]:

ex6_step2_chain = (
    ex6_plan_prompt
    | llm
    | ex6_plan_parser
)


In [11]:

weekly_plan = ex6_step2_chain.invoke({
    "fitness_details": fitness_details.model_dump_json(indent=2)
})

print(weekly_plan.model_dump_json(indent=2))


{
  "plan": [
    {
      "day": "Day 1",
      "focus": "Upper Body Strength",
      "exercises": [
        "Dumbbell Floor Press (3 sets of 8–10 reps)",
        "One‑Arm Dumbbell Row (3 sets of 8–10 reps per side)",
        "Dumbbell Shoulder Press (3 sets of 8–10 reps)",
        "Dumbbell Bicep Curl (3 sets of 10–12 reps)",
        "Dumbbell Triceps Kickback (3 sets of 10–12 reps per side)"
      ],
      "duration_minutes": 55,
      "notes": "Warm‑up: 5‑min dynamic arm circles, shoulder rolls, and light band pull‑aparts. Rest 90‑120 s between sets. Keep the back flat on the floor during presses and rows. Finish with 5‑min stretch focusing on chest, shoulders, and triceps."
    },
    {
      "day": "Day 2",
      "focus": "Lower Body Strength",
      "exercises": [
        "Goblet Squat (3 sets of 10–12 reps)",
        "Dumbbell Walking Lunge (3 sets of 10 reps per leg)",
        "Dumbbell Romanian Deadlift (3 sets of 10–12 reps)",
        "Standing Calf Raise with Dumbbells (3 se

In [12]:

def create_weekly_workout_plan(fitness_goal_text, days_per_week):
    # Step 1: Extract structured fitness details
    fitness_details = ex6_step1_chain.invoke({
        "fitness_goal_text": fitness_goal_text,
        "days_per_week": days_per_week
    })

    # Step 2: Generate the weekly workout plan
    weekly_plan = ex6_step2_chain.invoke({
        "fitness_details": fitness_details.model_dump_json(
            indent=2
        )
    })

    # Return both intermediate and final results
    return {
        "fitness_details": fitness_details,
        "weekly_plan": weekly_plan
    }


In [13]:

result = create_weekly_workout_plan(
    fitness_goal_text=(
        "I am a beginner who wants to build muscle at home. "
        "I have a pair of dumbbells."
    ),
    days_per_week=4
)

print("EXTRACTED FITNESS DETAILS:")
print(result["fitness_details"].model_dump_json(indent=2))

print("\nWEEKLY WORKOUT PLAN:")
print(result["weekly_plan"].model_dump_json(indent=2))


EXTRACTED FITNESS DETAILS:
{
  "goal_type": "Build muscle",
  "fitness_level": "Beginner",
  "days_available": 4,
  "equipment_access": "pair of dumbbells"
}

WEEKLY WORKOUT PLAN:
{
  "plan": [
    {
      "day": "Day 1",
      "focus": "Upper Body Strength",
      "exercises": [
        "Dumbbell Floor Press",
        "One‑Arm Dumbbell Row",
        "Dumbbell Shoulder Press",
        "Dumbbell Bicep Curl",
        "Dumbbell Tricep Kickback"
      ],
      "duration_minutes": 55,
      "notes": "Warm‑up: 5‑min light cardio (jumping jacks or marching in place) + dynamic stretches for chest, shoulders, and back. Perform 3 sets of 8–10 reps per exercise, resting 90 seconds between sets. Keep the core tight and avoid arching the back during rows. Finish with 5 minutes of static stretching for the upper body."
    },
    {
      "day": "Day 2",
      "focus": "Lower Body Strength",
      "exercises": [
        "Goblet Squat",
        "Dumbbell Walking Lunge",
        "Romanian Deadlift",
  